## Embedding Script

*(Config like `MODEL_NAME`, `BATCH_SIZE`, `EMB_DIR` comes from `00_config.ipynb`.)*

In [ ]:
import time
import math, os
import psutil
import numpy as np
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, models


EMB_DIR.mkdir(parents=True, exist_ok=True)

EMB_PATH     = EMB_DIR/"emb.npy"
IDS_PATH     = EMB_DIR/"ids.npy"

os.makedirs(os.path.dirname(EMB_PATH), exist_ok=True)
os.makedirs(os.path.dirname(IDS_PATH), exist_ok=True)
# -------------------------------------------

# ---- Validate input DataFrame ----
required_cols = {"passage_id", "doc_id", "text"}
missing = required_cols - set(PASSAGES_DF.columns)
if missing:
    raise KeyError(f"passages DataFrame missing required columns: {missing}")

# Drop rows with empty/NaN chunk_text (keep order stable)
df = PASSAGES_DF.copy()
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"] != ""].sort_values("passage_id").reset_index(drop=True)
if df.empty:
    raise ValueError("No non-empty passages to embed.")


texts = [f"passage: {t}" for t in df["text"].tolist()]
df["passage_id"] = df["passage_id"].astype("string")

# --- canonical passages used for embedding/indexing ---
df = df.sort_values("passage_id").reset_index(drop=True)

df["faiss_id"] = np.arange(len(df), dtype=np.int64)

CANONICAL_PASSAGES_PATH = EMB_DIR / "passages_canonical.parquet"
ID_MAP_PATH = EMB_DIR / "id_map.csv"

df.to_parquet(CANONICAL_PASSAGES_PATH, index=False)
df[["faiss_id", "passage_id"]].to_csv(ID_MAP_PATH, index=False)

# ids used for add_with_ids
ids = df["faiss_id"].values.astype(np.int64)

print("Saved canonical passages:", CANONICAL_PASSAGES_PATH)
print("Saved id_map:", ID_MAP_PATH)


In [9]:
from pathlib import Path
import os, time, json
import numpy as np
from sentence_transformers import SentenceTransformer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Load model
model = SentenceTransformer(MODEL_NAME)

# Paths (use the same ones defined earlier)
EMB_PATH  = Path(EMB_DIR / "emb.npy")
IDS_PATH  = Path(EMB_DIR / "ids.npy")
META_PATH = Path(EMB_DIR / "meta.json")

# IMPORTANT: build texts from the canonical df so ids/texts align 1:1
texts = [f"passage: {t}" for t in df["text"].astype(str).tolist()]
ids   = df["faiss_id"].values.astype(np.int64)

N = len(texts)
assert N == len(ids), f"Mismatch: N texts {N} vs ids {len(ids)}"

D = model.get_sentence_embedding_dimension()
dtype = np.dtype(EMB_DTYPE)

if EMB_PATH.exists() and IDS_PATH.exists() and META_PATH.exists():
    print("Embeddings already exist — skipping embedding step.")
    elapsed = 0.0  # so later prints don't crash
else:
    # Remove stale files (Windows-friendly)
    for p in [EMB_PATH, IDS_PATH, META_PATH]:
        try:
            if p.exists():
                p.unlink()
        except PermissionError:
            raise RuntimeError(f"{p} appears to be in use. Close readers and retry.")

    # Create memmap
    Xm = np.lib.format.open_memmap(
        str(EMB_PATH),
        mode="w+",
        dtype=dtype,
        shape=(N, D),
    )

    print(f"Memmap ready at {EMB_PATH} with shape {Xm.shape} and dtype {Xm.dtype}.")

    t0 = time.time()
    w = 0

    for i in range(0, N, BATCH_SIZE):
        batch = texts[i:i + BATCH_SIZE]
        vecs = model.encode(
            batch,
            batch_size=BATCH_SIZE,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype(dtype, copy=False)

        n = vecs.shape[0]
        Xm[w:w+n, :] = vecs
        w += n

    elapsed = time.time() - t0
    assert w == N, f"Wrote {w} vectors but expected {N}"

    # Flush + close memmap (important on Windows)
    Xm.flush()
    del Xm

    # Save ids + metadata
    np.save(IDS_PATH, ids)

    with open(META_PATH, "w", encoding="utf-8") as f:
        json.dump(
            {
                "model": MODEL_NAME,
                "dim": int(D),
                "n_vecs": int(N),
                "dtype": str(dtype),
                "normalize": True,
                "batch_size": int(BATCH_SIZE),
                "time_sec": round(elapsed, 2),
                "throughput_passages_per_sec": round(N / elapsed, 1) if elapsed > 0 else None,
            },
            f,
            indent=2,
        )

print(f"Embeddings: {(N, D)} saved → {EMB_PATH}")
print(f"IDs saved → {IDS_PATH}")
print(f"Meta saved → {META_PATH}")
print(f"Time: {elapsed:.2f}s | Throughput: {(N/elapsed):.1f}/s" if elapsed > 0 else "Time: skipped")


Embeddings already exist — skipping embedding step.
Embeddings: (10938, 384) saved → C:\Users\Harman\Documents\GISMA\Term_4\Thesis_Redo\Thesis_Implementation\rag_env\Project\rag-laptop\embeddings\intfloat_e5-small-v2\chunk_w512_ov0.25\emb.npy
IDs saved → C:\Users\Harman\Documents\GISMA\Term_4\Thesis_Redo\Thesis_Implementation\rag_env\Project\rag-laptop\embeddings\intfloat_e5-small-v2\chunk_w512_ov0.25\ids.npy
Meta saved → C:\Users\Harman\Documents\GISMA\Term_4\Thesis_Redo\Thesis_Implementation\rag_env\Project\rag-laptop\embeddings\intfloat_e5-small-v2\chunk_w512_ov0.25\meta.json
Time: skipped
